# 随机 test screening：joint model 与 LIDC GT 对照

从 fold 0 的 **testing split** 均匀随机抽取一个 screening，使用训练完成的 **V3 / epoch 18** checkpoint。
随机种子为 **422**，固定种子是为了复现这次抽样；抽样不参考预测或 GT 评分。

使用 `back_prop.evaluate.eval_package` 的函数展示：

1. 整个 screening 的初始 VISTA 分割与最终分割，各自对照官方读者 consensus GT。
2. 每个物理结节的预测/GT 分割，保留漏检及未匹配预测。
3. 两个 DataFrame：官方结节的 semantic features / malignancy 均值，以及模型候选的特征、恶性概率与官方结节匹配。

本 notebook 保存执行后的图表。推理结果也保存在旁边的 `try_results/`，再次打开或 Run All 可以复用结果，无需 GPU。
修改 checkpoint 或 seed 后，如果本机没有 GPU，会自动提交单卡 Slurm 作业，并等待结果。默认 kernel 使用已有的 **Python (py311)**；
推理由已有 **pro6000** Python 环境执行，避免要求 notebook kernel 安装 MONAI。

In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

%matplotlib inline

# 兼容从仓库根目录或当前 notebook 文件夹启动 Jupyter。
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'back_prop/evaluate/eval_package').is_dir()),
    Path('/home/users/yj255/code/imaging_feature'),
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from back_prop.evaluate.eval_package import (
    load_test_cases, load_result,
    screening_segmentation, nodule_segmentation,
    plot_screening_segmentation_3d, plot_nodule_segmentation,
)
from back_prop.evaluate.eval_package.annotations import case_key
from back_prop.evaluate.eval_package.types import SEMANTIC_NAMES

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 110)

SEED = 4
CHECKPOINT = Path('/usr/project/rudinlab/datasets/LIDC_IDRI/joint_model/model_v3_runs/12616464/epoch_018.pt')
MANIFEST = ROOT / 'vista3D/data/folds/fold_0.json'
INFERENCE_PYTHON = Path('/home/users/yj255/.local/miniforge3/envs/pro6000/bin/python')
NOTEBOOK_DIR = ROOT / 'back_prop/evaluate/eval_package'
OUTPUT_ROOT = NOTEBOOK_DIR / 'try_results' / f'v3_epoch018_seed{SEED}'
FORCE_INFERENCE = False
MASK_THRESHOLD = 0.5
MINIMUM_IOU = 0.1

print('Repository:', ROOT)
print('Checkpoint:', CHECKPOINT)

## 随机抽取一个 test screening

从按 patient / scan 排序后的整个 test split 中抽取一例；不按结节数量、malignancy 或模型表现筛选。
`load_test_cases` 同时检查 testing 与 training 的患者没有重叠。

In [ ]:
test_cases = sorted(load_test_cases(MANIFEST, split='testing'),
                    key=lambda c: (c['patient_id'], int(c['scan_id'])))
selected_case = random.Random(SEED).choice(test_cases)
CASE_ID = case_key(selected_case)
CASE_OUTPUT = OUTPUT_ROOT / CASE_ID

display(pd.DataFrame([{
    'random_seed': SEED,
    'test_screenings': len(test_cases),
    'selected_screening': CASE_ID,
    'patient_id': selected_case['patient_id'],
    'scan_id': selected_case['scan_id'],
    'CT_path': selected_case['image'],
}]))

## 推理一次，并读取配对的预测 / GT

已有缓存时直接读取；没有缓存时，先检查推理环境的 CUDA 可用性。
本机有 GPU 则直接推理；当前是 CPU 节点时自动提交 `rudin` 分区的单张 A6000 GPU 作业，并显示 job ID、排队/运行状态和日志目录。

修改 seed 也可以直接运行，无需把 notebook kernel 移到 GPU 上。中断等待不会取消已提交作业；重跑本单元格会继续等待同一作业。
模型只接收 CT / HU，不使用 GT 定位，不更新 FasterRisk bank。
默认保持模型 objectness 阈值，分割阈值为 0.5，匹配 IoU 阈值为 0.1。

In [ ]:
from back_prop.evaluate.eval_package.runner import ensure_case_result

CASE_OUTPUT = ensure_case_result(
    checkpoint=CHECKPOINT, manifest=MANIFEST, case_id=CASE_ID,
    output_root=OUTPUT_ROOT, python=INFERENCE_PYTHON, root=ROOT,
    force=FORCE_INFERENCE, mask_threshold=MASK_THRESHOLD, minimum_iou=MINIMUM_IOU,
    execution='auto', partition='rudin', account='rudin', gres='gpu:a6000:1',
)

result = load_result(CASE_OUTPUT)
assert result.case_id == CASE_ID
assert Path(result.metadata['checkpoint']).resolve() == CHECKPOINT.resolve(), '缓存的 checkpoint 不同'
assert Path(result.metadata['manifest']).resolve() == MANIFEST.resolve(), '缓存的 manifest 不同'
assert result.metadata['split'] == 'testing'
assert result.metadata['mask_threshold'] == MASK_THRESHOLD
assert result.minimum_iou == MINIMUM_IOU
assert result.metadata['ground_truth_used_for_prediction'] is False
assert result.metadata['bank_unchanged'] is True

display(pd.DataFrame([{
    'case_id': result.case_id,
    'architecture': result.metadata['architecture'],
    'checkpoint_epoch': result.metadata['epoch'],
    'CT_XYZ_shape': result.shape,
    'predicted_nodules': len(result.predictions),
    'official_GT_nodules': len(result.ground_truth),
    'object_threshold': result.metadata['object_threshold'],
    'mask_threshold': result.metadata['mask_threshold'],
    'GT_used_for_inference': result.metadata['ground_truth_used_for_prediction'],
}]))

## 1. 整个 screening 的 segmentation（三维）

采用与 `visual.ipynb` 的 cell 2 相同的 marching cubes + Plotly Mesh3d 表面叠加方式，分别展示初始 VISTA screening 和最终 refined-nodule mask 并集。
**红色为模型预测，蓝色为官方读者 consensus GT**；可以拖动旋转、滚轮缩放、点击图例切换显示。

展示整个 3D mask 的全部前景，坐标通过 CT affine 转换为实际空间的 R / A / S 毫米坐标，并保持物理比例。
GT 仍使用每个物理结节至少 `min(2, 读者数)` 票的 consensus；Dice / IoU 使用完整 3D 体积。
空预测 mask 会明确标注。每张交互图另存为可单独打开的 HTML。

In [ ]:
scan_metric_rows = []
for stage in ('screening', 'final'):
    comparison = screening_segmentation(result, stage=stage)
    scan_metric_rows.append({'stage': stage, **comparison['metrics']})
    del comparison
display(pd.DataFrame(scan_metric_rows).set_index('stage'))
if scan_metric_rows[-1]['predicted_voxels'] == 0:
    display(Markdown(
        f'**本例最终二值分割为空。模型通过 objectness 阈值的候选 ROI 有 {len(result.predictions)} 个，'
        f'官方 GT 有 {len(result.ground_truth)} 个物理结节。候选的存在不代表其分割 mask 非空。**'
    ))

for stage in ('screening', 'final'):
    html_path = CASE_OUTPUT / f'{stage}_3d.html'
    fig = plot_screening_segmentation_3d(result, stage=stage, save_path=html_path)
    fig.show(renderer='plotly_mimetype+notebook')
    print('Interactive 3D HTML:', html_path)

## 2. 分 nodule 的 segmentation

根据 3D mask IoU 做一对一匹配；`nodule_id` 为 GT 物理结节 ID，`query_id` 为模型预测 ID。
`matched` 表示匹配成功，`missed_gt` 表示没有匹配的预测，`unmatched_prediction` 表示没有对应 GT。
下面展示所有 GT 结节和所有未匹配预测，不挑选成功案例。

In [ ]:
segmentation_rows = nodule_segmentation(result)
nodule_table = pd.DataFrame([
    {key: row[key] for key in ('nodule_id', 'query_id', 'status', 'gt_mask_empty')}
    | row['metrics']
    for row in segmentation_rows
])
display(nodule_table)
if not result.predictions:
    display(Markdown('**本例在默认 objectness 阈值下没有输出预测结节；下面仍展示全部 GT。**'))
if not segmentation_rows:
    display(Markdown('本例没有预测结节，也没有官方 GT 结节。'))

for row in segmentation_rows:
    if row['nodule_id'] is not None:
        selector = {'nodule_id': row['nodule_id']}
        name = f"nodule_id_{row['nodule_id']}"
    else:
        selector = {'query_id': row['query_id']}
        name = f"query_id_{row['query_id']}"
    fig = plot_nodule_segmentation(result, **selector, save_path=CASE_OUTPUT / f'{name}.png')
    display(fig)
    plt.close(fig)

## 3. 官方结节与模型输出（两个 DataFrame）

每行对应一个结节，表格左侧标明总数。

- **第一张表 `official_nodules_df`**：按官方物理结节 `nodule_id` 汇总。六个 semantic features 和 `malignancy` 均为该结节所有官方读者评分的算术均值（1–5 分）。
- **第二张表 `model_nodules_df`**：每行是一个通过 objectness 阈值的模型候选 `query_id`，包含六个 semantic features 的期望评分（1–5 分）、`malignancy_probability`（0–1 概率）以及 `matched_gt_nodule_id`，对应第一张表的 `nodule_id`。

匹配使用与第 2 部分相同的 3D mask IoU 一对一规则和 `MINIMUM_IOU` 阈值；未匹配时，官方 ID 和 `match_iou` 为空。
模型候选数量包含最终 mask 为空的候选，`mask_voxels` 显示其分割体素数；空 mask 不会匹配到官方结节。
官方评分来自 `all_ct_annotations.csv`，由 `nodule_iden.csv` 归并物理结节；模型的恶性概率与官方 1–5 分 malignancy 均值是不同量纲。


In [ ]:
# 官方结节：保留所有物理结节，包括没有匹配预测的结节。
official_records = []
for nodule in result.ground_truth:
    means = nodule.means
    official_records.append({
        'nodule_id': nodule.nodule_id,
        **{feature: means.get(feature) for feature in SEMANTIC_NAMES},
        'malignancy': means.get('malignancy'),
    })
official_nodules_df = pd.DataFrame(official_records, columns=[
    'nodule_id', *SEMANTIC_NAMES, 'malignancy',
]).astype({'nodule_id': 'Int64'})


# 模型候选：复用分割的匹配规则，不强制为未匹配预测指定官方结节。
match_by_prediction = {
    row['prediction_index']: row for row in nodule_segmentation(result)
    if row['prediction_index'] is not None
}
model_records = []
for index, nodule in enumerate(result.predictions):
    match = match_by_prediction[index]
    model_records.append({
        'query_id': nodule.query_id,
        **{feature: nodule.semantic_features.get(feature) for feature in SEMANTIC_NAMES},
        'malignancy_probability': nodule.malignancy_probability,
        'matched_gt_nodule_id': match['nodule_id'],
        'match_iou': match['metrics']['iou'] if match['status'] == 'matched' else np.nan,
        'mask_voxels': match['metrics']['predicted_voxels'],
    })
model_nodules_df = pd.DataFrame(model_records, columns=[
    'query_id', *SEMANTIC_NAMES, 'malignancy_probability',
    'matched_gt_nodule_id', 'match_iou', 'mask_voxels',
]).astype({'query_id': 'Int64', 'matched_gt_nodule_id': 'Int64', 'mask_voxels': 'Int64'})


display(official_nodules_df)
display(model_nodules_df)


## 保存与来源记录

整个 screening 的三维图保存为独立 HTML；逐结节分割保存为 PNG。`result.json` 包含完整预测、读者评分、匹配及来源信息，
`arrays.npz` 保存 HU、mask、概率和 affine；同时提供初始/最终分割以及 GT 的 NIfTI。
下面保存抽样信息、分割指标，以及第 3 部分的 `official_nodules.csv` 和 `model_nodules.csv`，不再额外显示表格。


In [ ]:
CASE_OUTPUT.mkdir(parents=True, exist_ok=True)
sampling_record = {
    'seed': SEED, 'sampling': 'uniform choice from sorted full testing split',
    'test_screenings': len(test_cases), 'selected_case_id': CASE_ID,
    'manifest': str(MANIFEST), 'checkpoint': str(CHECKPOINT),
    'mask_threshold': MASK_THRESHOLD, 'minimum_iou': MINIMUM_IOU,
}
(CASE_OUTPUT / 'sampling.json').write_text(json.dumps(sampling_record, ensure_ascii=False, indent=2) + '\n')
nodule_table.to_csv(CASE_OUTPUT / 'nodule_segmentation.csv', index=False)
official_nodules_df.to_csv(CASE_OUTPUT / 'official_nodules.csv', index=False)
model_nodules_df.to_csv(CASE_OUTPUT / 'model_nodules.csv', index=False)
